In [ ]:
# | default_exp transforms/monai/pixdim

# Imports

In [ ]:
# | export


from typing import Any

import torch
from monai.transforms import MapTransform

In [ ]:
# | export


class SetPixdimFromSpacing(MapTransform):
    def __init__(self, keys, spacing_keys: str | tuple[str, str, str]):
        """Set the pixdim/affine of MetaTensors from spacing values in the data dictionary.

        Args:
            keys: Keys of the MetaTensors whose affine will be updated.
            spacing_keys: Key(s) to retrieve spacing from the data dict. A dot-separated string
                (e.g. ``"metadata.spacing"``) for nested lookup of a single spacing vector, or a tuple
                of three such strings for individual per-axis spacing values (z, y, x).
        """
        super().__init__(keys, allow_missing_keys=False)

        self.spacing_keys = SetPixdimFromSpacing._parse_spacing_keys(spacing_keys)

    def __call__(self, data: dict[str, Any]):
        """Apply the spacing-derived affine to each keyed MetaTensor in ``data`` and return it.

        Args:
            data: Dictionary containing MetaTensors (at ``self.keys``) and spacing values
                (at ``self.spacing_keys``).
        """
        spacing = self._get_spacing(data)
        affine = self._spacing_to_affine(spacing)
        for key in self.keys:
            data[key].affine = affine
        return data

    @staticmethod
    def _parse_spacing_keys(spacing_keys: str | tuple[str, str, str]) -> list[str] | tuple:
        """Parse spacing keys into dot-split lists for nested dict lookup. Recurses for tuples."""
        if isinstance(spacing_keys, (list, tuple)):
            return tuple([SetPixdimFromSpacing._parse_spacing_keys(spacing_key) for spacing_key in spacing_keys])
        return spacing_keys.split(".")

    def _get_spacing(self, data: dict) -> torch.Tensor:
        """Retrieve the spacing from the data dict using the parsed spacing keys."""

        def get_value(root: dict, keys: list[str]):
            value = root[keys[0]]
            if len(keys) == 1:
                return root[keys[0]]
            return get_value(value, keys[1:])

        if isinstance(self.spacing_keys, tuple):
            # All three values were given separately
            spacing = [get_value(data, spacing_key) for spacing_key in self.spacing_keys]
        else:
            # A single key holds the entire spacing
            spacing = get_value(data, self.spacing_keys)
        spacing = torch.tensor(spacing)

        return spacing

    @staticmethod
    def _spacing_to_affine(spacing) -> torch.Tensor:
        """Convert spacing tensor to affine matrix according to Metatensor notation.

        Args:
            spacing: 1-D tensor of shape ``(3,)`` with voxel spacing, or ``None`` to default to ones.

        Returns:
            A ``(4, 4)`` diagonal affine matrix with spacing on the diagonal.
        """
        if spacing is None:
            spacing = torch.ones(3)
        return torch.diag(torch.concatenate([spacing, torch.zeros(1)]))

In [ ]:
from monai.data import MetaTensor

datapoint = {
    "image": MetaTensor(torch.rand(10, 10, 10)),
    "image_metadata": {
        "spacing": torch.tensor([0.1, 0.2, 0.3]),
        "spacing_z": 0.4,
        "spacing_y": torch.tensor(0.5),
        "spacing_x": 0.6,
    },
    "spacing": [0.7, 0.8, 0.9],
    "spacing_z": torch.tensor(1.0),
    "spacing_y": 1.1,
    "spacing_x": torch.tensor(1.2),
}

transform = SetPixdimFromSpacing("image", "spacing")
output = transform(datapoint)
print(output["image"].pixdim)

transform = SetPixdimFromSpacing("image", "image_metadata.spacing")
output = transform(datapoint)
print(output["image"].pixdim)

transform = SetPixdimFromSpacing("image", ("spacing_z", "spacing_y", "spacing_x"))
output = transform(datapoint)
print(output["image"].pixdim)

transform = SetPixdimFromSpacing(
    "image", ("image_metadata.spacing_z", "image_metadata.spacing_y", "image_metadata.spacing_x")
)
output = transform(datapoint)
print(output["image"].pixdim)

tensor([0.7000, 0.8000, 0.9000], dtype=torch.float64)
tensor([0.1000, 0.2000, 0.3000], dtype=torch.float64)
tensor([1.0000, 1.1000, 1.2000], dtype=torch.float64)
tensor([0.4000, 0.5000, 0.6000], dtype=torch.float64)


/tmp/ipykernel_608421/638181899.py:44: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  spacing = torch.tensor(spacing)


# nbdev

In [ ]:
!nbdev_export